# FELIX hit dossier — one page of diagnostics per significant locus

Assembles everything needed to judge whether each retained hit is real, short of the read pileups, and exports it for
the review plate (`manuscript_figures/figS_hit_dossier.py`, one page per locus).

**What a locus is here.** A hit the screen retained (`verdict == 'keep'`: none of spa_fallback, no_case_carriers,
impossible_p, excess_het fired) with at least 10 case carriers. Hits of one phenotype within 500 kb are one locus.
That is the same rule the Figure 3 table and PheWAS panel count, enforced by a test, so the plate reviews exactly the
hits the figure reports.

**What this notebook writes**

| file | one row per | cost |
|---|---|---|
| `felix_dossier_loci.tsv` | locus: identity, lead, arms, phenotype calibration, window | free |
| `felix_dossier_leads.tsv` | locus x arm: the lead marker's full per-ancestry block | free (harvest cache) |
| `felix_dossier_context.tsv` | locus: cytoband, segmental duplications, tandem repeat, genes | UCSC API |
| `felix_dossier_genes.tsv` | gene in a locus window, for the regional panel's gene track | UCSC API |
| `felix_dossier_region.<locus>.tsv.gz` | marker near the locus, thinned, all arms | **one stream per run** |

Everything except the region files comes out of the harvest cache that `felix_08_hit_sweep` already built, so sections
1-6 take about a minute. Section 7 is the expensive one: it streams each phenotype's `merged.felix.tsv` (0.5-27 GB)
once for all of that phenotype's loci.

**Run order.** `00_sync_repo` (so the bucket copies of `felix_phenotype_report.py`, `felix_hit_dossier.py` and
`extract_region.py` are current), then `felix_08_hit_sweep` (builds `harvest_cache/`), then this notebook.

## 1 · Setup

In [ ]:
import os, json, subprocess, sys, warnings, importlib
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 80)

ETYPE    = "flare_lai_prod_set"
ID_COL   = f"entity:{ETYPE}_id"
TAR_COL  = "run_summary_tar"
NUM_ANCS = 5
STORE    = "harvest_cache"      # built by felix_08; shared, not rebuilt here unless it is missing
REGION_DIR = Path("dossier_regions")   # compact per-locus region files land here

WS_NS   = os.environ.get("WORKSPACE_NAMESPACE")
WS_NAME = os.environ.get("WORKSPACE_NAME")
BUCKET  = os.environ.get("WORKSPACE_BUCKET")
print("workspace:", WS_NS, "/", WS_NAME)
assert WS_NS and WS_NAME, "set WS_NS / WS_NAME by hand if the env vars are absent"

# ALWAYS re-copy the modules. Several notebooks write these filenames into this working directory, and an older sync
# leaves an older copy, so a "fetch only if absent" guard silently imports stale code.
MODULES = ["felix_phenotype_report.py", "felix_hit_dossier.py", "extract_region.py",
           "pull_site_fields.py"]
for m in MODULES:
    subprocess.run(["gsutil", "cp", f"{BUCKET}/felix/scripts/{m}", m], check=True)

importlib.invalidate_caches()
for m in ("felix_phenotype_report", "felix_hit_dossier", "extract_region", "pull_site_fields"):
    sys.modules.pop(m, None)
import felix_phenotype_report as fpr
import felix_hit_dossier as fhd
import extract_region as er
import pull_site_fields as ps

REQUIRED = {
    "felix_phenotype_report": (fpr, ["sweep", "harvest", "read_harvest", "hit_report", "calibration_report",
                                     "excess_het", "corroborated", "recurrence"]),
    "felix_hit_dossier":      (fhd, ["dossier_loci", "lead_rows", "annotate_context", "region_plan",
                                     "compact_region", "checks", "verdict"]),
    "extract_region":         (er,  ["extract_many", "read_windows", "split_windows"]),
    "pull_site_fields":       (ps,  ["query_sites", "call_rate_by_group", "imputed_fraction_by_group",
                                     "case_control_map", "available_format_tags"]),
}
for name, (mod, funcs) in REQUIRED.items():
    missing = [f for f in funcs if not hasattr(mod, f)]
    if missing:
        raise RuntimeError(
            f"{name}.py is missing {missing}.\n"
            f"The copy at {BUCKET}/felix/scripts/{name}.py predates the dossier work. "
            f"Re-run 00_sync_repo (it rsyncs felix/scripts/ to the bucket from origin/main), then re-run this cell.")
print("modules ok:", ", ".join(REQUIRED))

## 2 · Run table

In [ ]:
from firecloud import api as fapi

def entities_to_df(ns, name, etype):
    r = fapi.get_entities(ns, name, etype); r.raise_for_status()
    rows = []
    for e in r.json():
        rec = {f"entity:{etype}_id": e["name"]}
        for k, v in e.get("attributes", {}).items():
            if isinstance(v, dict) and "items" in v:
                v = list(v["items"])
            rec[k] = v
        rows.append(rec)
    return pd.DataFrame(rows)

runs = entities_to_df(WS_NS, WS_NAME, ETYPE).sort_values(ID_COL).reset_index(drop=True)
done = runs[runs[TAR_COL].notna()].copy()
print(f"{len(runs)} rows in {ETYPE}; {len(done)} have {TAR_COL}")

## 3 · Screen every run, then reduce the hits to loci

Re-runs the same screen as `felix_08` off the shared harvest cache (seconds when the cache is warm, a few minutes
cold), then applies the locus rule. The counts printed here are the ones the plate will show.

In [ ]:
%%time
cal, hits, failures = fpr.sweep(done, ID_COL, TAR_COL, store=STORE, num_ancs=NUM_ANCS, workers=8)
print(f"{len(cal)} run x arm calibration rows; {len(hits)} genome-wide hits; {len(failures)} harvest failures")
if len(failures):
    display(failures)

loci = fhd.dossier_loci(hits)
kept = fhd.retained(hits)
print(f"\n{len(hits)} reported hits -> {int((hits.verdict == 'keep').sum())} retained by the screen "
      f"-> {len(kept)} with at least {fhd.MIN_CASE_CARRIERS} case carriers -> {len(loci)} loci "
      f"across {loci.phenotype.nunique()} phenotypes")
print("\nby lead arm:", loci.lead_arm.value_counts().to_dict())
print("arms per locus:", loci.arms.value_counts().to_dict())
display(loci.head(10))

## 4 · Lead statistics, with every per-ancestry column

The lead of each arm that retained the locus, matched inside the locus window on the arm's own `top_hits.tsv`. These
are the columns the effect forest, the genotype block and the joint-test coherence check are computed from, and they
are already in the harvest cache -- no streaming.

In [ ]:
%%time
top = {}
for run_dir in sorted(Path(STORE).iterdir()):
    if not run_dir.is_dir() or run_dir.name.startswith("_"):
        continue
    b = fpr.read_harvest(run_dir)
    for arm in fpr.ARMS:
        th = b["arms"].get(arm, {}).get("top_hits")
        if th is not None and len(th):
            top[(run_dir.name, arm)] = th
print(f"{len(top)} run x arm top_hits tables in the cache")

leads = fhd.lead_rows(loci, top)
found = leads[leads.found.astype(bool)]
print(f"{len(leads)} locus x arm rows; {len(found)} matched a top_hits row, {len(leads) - len(found)} did not")
if len(leads) > len(found):
    display(leads[~leads.found.astype(bool)][["locus_id", "arm", "note"]])

# Sanity: the lead p a dossier row reports must be the p the sweep reported for that locus and arm.
chk = found.merge(loci[["locus_id", "lead_arm", "p"]], on="locus_id")
chk = chk[chk.arm == chk.lead_arm]
delta = (pd.to_numeric(chk["P_cct_admixed_c"], errors="coerce") / chk["p"] - 1).abs()
print(f"lead p agrees with the sweep for {int((delta < 1e-6).sum())}/{len(chk)} loci "
      f"(largest relative difference {delta.max():.2e})")
display(found[["locus_id", "arm", "MarkerID", "P_cct_admixed_c", "P_het_admixed_c", "P_hom_admixed_c"]].head(10))

## 5 · Phenotype context

Each locus carries its phenotype's case count and per-arm genomic-control lambda, so the page can say whether the
scan that produced the hit was itself calibrated. `lambda_gc` below about 0.9 means the arm is deflated and no hit
screen will change that.

In [ ]:
wide = cal.pivot_table(index="run", columns="arm", values=["lambda_gc", "n_cases", "N_eff"], aggfunc="first")
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()
keep_cols = ["run"] + [c for c in wide.columns if c.startswith("lambda_gc_")]
ctx_cols = wide[keep_cols].copy()
ctx_cols["n_cases"] = wide[[c for c in wide.columns if c.startswith("n_cases_")]].max(axis=1)
ctx_cols["N_eff"] = wide[[c for c in wide.columns if c.startswith("N_eff_")]].max(axis=1)

loci = loci.merge(ctx_cols, on="run", how="left")
print("lambda of the lead arm, across the dossier loci:")
lead_lambda = loci.apply(lambda r: r.get(f"lambda_gc_{r.lead_arm}"), axis=1)
print(lead_lambda.describe().round(3).to_string())
print(f"\nloci whose lead arm has lambda below 0.90: {int((lead_lambda < 0.90).sum())}")
display(loci[["locus_id", "phenotype", "n_cases", "lambda_gc_main", "lambda_gc_ru", "lambda_gc_ru_biallelic"]].head(10))

## 6 · Genomic context

Cytoband, segmental duplications, tandem repeats and genes from the UCSC hg38 API — the questions "is this
pericentromeric", "is it in a duplication" and "what gene is it in" that a reviewer asks before anything else. About
four requests per locus.

In [ ]:
%%time
context, genes, tracks = fhd.annotate_context(loci)
bad = context[context.error != ""]
print(f"{len(context)} loci annotated; {len(bad)} lookups failed")
if len(bad):
    display(bad[["locus_id", "error"]])
print(f"pericentromeric: {int(context.pericentromeric.sum())}; "
      f"inside a segmental duplication: {int(context.in_segdup.sum())}; "
      f"inside a tandem repeat: {int(context.in_tandem_repeat.sum())}; "
      f"{len(genes)} gene records and {len(tracks)} repeat/duplication features in the windows")
display(context[["locus_id", "cytoband", "in_segdup", "segdup_frac", "in_tandem_repeat", "gene",
                 "nearest_gene", "nearest_gene_bp"]].head(10))

## 7 · Region extracts — the expensive step

The only diagnostic that needs the full results: does the peak have neighbours? One `gsutil cat` pass per run and arm
covers every locus of that phenotype, and each window is immediately reduced to the ~26 columns the regional panel
reads, keeping every marker below p 1e-3 plus a 3,000-row background sample. The raw cut is deleted as soon as it is
compacted, so peak disk stays in the hundreds of MB rather than ~1 GB.

Re-running is cheap for loci already done: set `REDO = True` to force them.

In [ ]:
%%time
REDO = False
REGION_DIR.mkdir(exist_ok=True)
plan = fhd.region_plan(loci)
by_id = {r.locus_id: r for _, r in loci.iterrows()}   # Series, so compact_region can read ["window_start"]
print(f"{sum(len(v) for v in plan.values())} windows across {len(plan)} runs")

written, skipped, problems = [], [], []
for i, (run, spans) in enumerate(sorted(plan.items()), 1):
    ids = [lid for s in spans for lid in s["locus_id"].split(",")]
    if not REDO and all(fhd.region_path(REGION_DIR, lid, arm).exists() for lid in ids for arm in fpr.ARMS):
        skipped += ids
        continue
    row = done[done[ID_COL] == run]
    if len(row) != 1:
        problems.append(dict(run=run, error=f"{len(row)} rows in the run table"))
        continue
    root = row[TAR_COL].iloc[0].split("/call-Bundle")[0]
    print(f"[{i}/{len(plan)}] {run}: {len(spans)} window(s)", flush=True)
    for arm in fpr.ARMS:
        objs = er.list_call(root, er.ARM_CALL[arm])
        uri = er.find_merged(objs)
        if not uri:
            problems.append(dict(run=run, arm=arm, error=f"no merged table under call-{er.ARM_CALL[arm]}"))
            continue
        targets = [(str(s["chrom"]), s["start"], s["end"], Path(f"_raw.{arm}.{j}.tsv"))
                   for j, s in enumerate(spans)]
        try:
            counts = er.extract_many(uri, targets)
        except Exception as exc:
            problems.append(dict(run=run, arm=arm, error=f"{type(exc).__name__}: {exc}"))
            continue
        for (span, (_, _, _, raw), n) in zip(spans, targets, counts):
            for lid in span["locus_id"].split(","):
                out = fhd.region_path(REGION_DIR, lid, arm)
                try:
                    k = fhd.compact_region(raw, out, by_id[lid])
                    written.append(dict(locus_id=lid, arm=arm, raw_rows=n, kept_rows=k,
                                        kb=round(out.stat().st_size / 1024, 1)))
                except Exception as exc:
                    problems.append(dict(run=run, arm=arm, locus_id=lid, error=f"{type(exc).__name__}: {exc}"))
            raw.unlink(missing_ok=True)

w = pd.DataFrame(written)
print(f"\nwrote {len(w)} compact region files ({w.kb.sum() / 1024:.1f} MB total)" if len(w) else "\nnothing written")
print(f"skipped {len(set(skipped))} loci already done; {len(problems)} problems")
if problems:
    display(pd.DataFrame(problems))
if len(w):
    display(w.head(10))

## 8 · Call rate at the lead sites

The one diagnostic FELIX's own output cannot give. `phase_vcfs` is the ligated phased callset, where phasing has filled
every genotype, so `MissingRate` is 0 at every record — not a clean call rate, an absent one.

This pulls the genotypes back from the indexed callsets directly: one `bcftools query -R` per VCF, one index seek per
lead, nothing localised and nothing merged. Two sources per locus — the **pre-phasing** DeepVariant/GLnexus VCF and the
**phased** `gt_vcf` the scan actually read — which give two numbers: the case/control call rate, and the share of the
genotypes the scan used that phasing *imputed* rather than observed. A difference between cases and controls in the
second is the failure mode a call-rate comparison alone cannot see.

Fill `PRE_PHASING_VCFS` with the per-chromosome DeepVariant/GLnexus URIs (the `GL_INTERVAL_set` table that
`propagate_annotations/wdl/AnnotateFlareGqDp.wdl` names as its `deepvariant_vcf`). Leave it empty and the cell reports
the phased call rate only, which is 1.0 everywhere by construction and says so.

In [ ]:
%%time
# htslib range-reads gs:// when the environment carries credentials; without these every query fails on auth.
os.environ.setdefault("GCS_OAUTH_TOKEN", subprocess.run(["gcloud", "auth", "print-access-token"],
                                                        capture_output=True, text=True).stdout.strip())
os.environ.setdefault("GCS_REQUESTER_PAYS_PROJECT", os.environ.get("GOOGLE_PROJECT", ""))

# The phased callset the scan read, per chromosome, straight off the entity table.
chrom_rows = entities_to_df(WS_NS, WS_NAME, "flare_lai_prod")
PHASED_VCFS = {str(r["chrom"]): r["gt_vcf"] for _, r in chrom_rows.iterrows() if r.get("gt_vcf")}
print(f"phased gt_vcf for {len(PHASED_VCFS)} chromosomes")

PRE_PHASING_VCFS = {}   # {'chr9': 'gs://.../chr9.glnexus.vcf.gz', ...}  <- fill this
if not PRE_PHASING_VCFS:
    print("PRE_PHASING_VCFS is empty: call rate will be reported from the phased callset only, "
          "which is 1.0 by construction. Fill it to get a real call rate.")

sites = fhd.lead_sites(loci)
sites["chrom_name"] = "chr" + sites.chrom.astype(str)
# pheno_cov is a PER-RUN input (the config passes ${this.pheno_cov}), not a global one: most runs share
# felix/phenotypes/wgs/pheno_cov.tsv, while a sex-restricted phenotype has its own file under
# felix/phenotypes/prod/. Mapping such a locus with the shared file would put the excluded participants back into
# the control set and quietly inflate its call-rate denominator, so each locus is mapped with its own run's file.
pheno_cov_by_run = {r[ID_COL]: str(r["pheno_cov"]).strip()
                    for _, r in done.iterrows() if str(r.get("pheno_cov", "")).strip()}
all_uris = set(pheno_cov_by_run.values())
needed = sorted({pheno_cov_by_run.get(r) for r in sites.run} - {None})   # only what the dossier's runs point at
orphans = sorted({r for r in sites.run if pheno_cov_by_run.get(r) is None})
print(f"{len(all_uris)} distinct pheno_cov across {len(pheno_cov_by_run)} runs; "
      f"the dossier's {sites.run.nunique()} runs need {len(needed)} of them"
      + (f"; {len(orphans)} run(s) have no pheno_cov in the table: {orphans[:3]}" if orphans else ""))

PHENO_DIR = Path("pheno_cov_files")
PHENO_DIR.mkdir(exist_ok=True)
pheno_by_uri = {}
for u in needed:
    local = PHENO_DIR / Path(u).name if u.startswith("gs://") else Path(u)
    if u.startswith("gs://") and not local.exists():
        subprocess.run(["gsutil", "cp", u, str(local)], check=True)
    pheno_by_uri[u] = pd.read_csv(local, sep="	", low_memory=False)
    have = sum(1 for ph in sites.phenotype.unique() if ph in pheno_by_uri[u].columns)
    print(f"  {Path(u).name}: {len(pheno_by_uri[u])} rows, {len(pheno_by_uri[u].columns)} columns, "
          f"{have}/{sites.phenotype.nunique()} dossier phenotypes present")

# The phased callset is only worth reading to VERIFY that phasing left no no-calls behind; it is 0 at every record
# of FELIX's input, so by default we take that as given and read the pre-phasing callset alone.
VERIFY_PHASED = False
sources = {"pre": PRE_PHASING_VCFS}
if VERIFY_PHASED:
    sources["post"] = PHASED_VCFS

# One probe per source before the loop: a gs:// URI needs an htslib built with libcurl AND GCS_OAUTH_TOKEN in the
# environment, and when either is missing every chromosome fails identically. Better to see htslib's own words once.
for label, mapping in sources.items():
    if mapping:
        uri = sorted(mapping.values())[0]
        print(f"{label}: probing " + uri.rsplit("/", 1)[-1])
        print("  " + ps.probe(uri, os.environ.get("GOOGLE_PROJECT")), flush=True)

pulled = {}
for label, mapping in sources.items():
    frames, failed = [], 0
    for chrom_name, uri in sorted(mapping.items()):
        want = sites[sites.chrom_name == chrom_name]
        if want.empty:
            continue
        try:
            got = ps.query_sites(uri, want, fields=("GT", "DP", "GQ", "RNC"),
                                 gcs_project=os.environ.get("GOOGLE_PROJECT"))
        except Exception as exc:
            failed += 1
            if failed == 1:                      # the first failure in full; the rest would repeat it
                print(f"  {label} {chrom_name} FAILED: {exc}")
            continue
        print(f"  {label} {chrom_name}: {len(got)} rows for {got.locus_id.nunique() if len(got) else 0} loci",
              flush=True)
        frames.append(got)
    if failed > 1:
        print(f"  {label}: {failed} chromosomes failed the same way")
    pulled[label] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
pulled.setdefault("post", pd.DataFrame())

# Each locus is its own phenotype, so the case/control split is per locus, not global.
rows = []
for _, S in sites.iterrows():
    pheno = pheno_by_uri.get(pheno_cov_by_run.get(S.run))
    if pheno is None:
        rows.append(dict(locus_id=S.locus_id, note=f"no pheno_cov for run {S.run}"))
        continue
    # pheno_cov names its columns with the underscore form (BI_164_6); the figure code and PhecodeX use the dotted
    # form (BI_164.6). They coincide only without a sub-code, so offer both and let it find whichever is there.
    candidates = [S.run.removeprefix("felix.prod."), S.phenotype]
    try:
        column = ps.phenotype_column(pheno, candidates)
        groups = ps.case_control_map(pheno, column)
    except KeyError:
        rows.append(dict(locus_id=S.locus_id, phenotype=S.phenotype,
                         note=f"none of {candidates} is a column in {Path(pheno_cov_by_run[S.run]).name}"))
        continue
    rec = dict(locus_id=S.locus_id, phenotype=S.phenotype, pheno_column=column, note="",
               n_case_total=sum(1 for v in groups.values() if v == "case"),
               n_control_total=sum(1 for v in groups.values() if v == "control"))
    for label in ("pre", "post"):
        df = pulled[label]
        sub = df[df.locus_id == S.locus_id] if len(df) else pd.DataFrame()
        cr = ps.call_rate_by_group(sub, groups)
        if len(cr):
            for k, v in cr.iloc[0].items():
                if k != "locus_id":
                    rec[f"{label}_{k}"] = v
    if len(pulled["pre"]):
        post_rows = (pulled["post"][pulled["post"].locus_id == S.locus_id]
                     if len(pulled["post"]) else None)   # None = take it that phasing filled every no-call
        imp = ps.imputed_fraction_by_group(pulled["pre"][pulled["pre"].locus_id == S.locus_id], post_rows, groups)
        if len(imp):
            for k, v in imp.iloc[0].items():
                if k != "locus_id":
                    rec[k] = v
    rows.append(rec)

callrate = pd.DataFrame(rows)
callrate.to_csv(fhd.CALLRATE_FILE, sep="	", index=False)
print(f"\nwrote {fhd.CALLRATE_FILE}: {len(callrate)} loci")
resolved = callrate.get("pheno_column")
if resolved is not None:
    print(f"phenotype column resolved for {int(resolved.notna().sum())}/{len(callrate)} loci")
notes = callrate[callrate.note.astype(str).str.len() > 0] if "note" in callrate.columns else pd.DataFrame()
if len(notes):
    print(f"{len(notes)} loci could not be mapped; first few:")
    display(notes[["locus_id", "phenotype", "note"]].head(5))
cols = [c for c in ("locus_id", "phenotype", "n_case_total", "pre_call_rate_case", "pre_call_rate_control",
                    "pre_fisher_p", "imputed_frac_case", "imputed_frac_control", "imputed_frac_diff")
        if c in callrate.columns]
if len(cols) > 2:
    display(callrate[cols].head(12))

## 8 · Export

In [ ]:
loci.to_csv(fhd.LOCI_FILE, sep="\t", index=False)
leads.to_csv(fhd.LEADS_FILE, sep="\t", index=False)
context.to_csv(fhd.CONTEXT_FILE, sep="\t", index=False)
genes.to_csv(fhd.GENES_FILE, sep="\t", index=False)
tracks.to_csv(fhd.TRACKS_FILE, sep="\t", index=False)
print("wrote:", sorted(p.name for p in Path(".").glob("felix_dossier_*.tsv")))
print(f"region files: {len(list(REGION_DIR.glob('*.tsv.gz')))}")

# A dry run of the checks, so the notebook says what the plate will say before anyone renders it.
rows = []
for _, L in loci.iterrows():
    lr = leads[(leads.locus_id == L.locus_id) & (leads.arm == L.lead_arm) & leads.found.astype(bool)]
    lead = lr.iloc[0] if len(lr) else None
    cx = context[context.locus_id == L.locus_id]
    reg_path = fhd.region_path(REGION_DIR, L.locus_id, L.lead_arm)
    region = pd.read_csv(reg_path, sep="\t") if reg_path.exists() else None
    flags = fhd.checks(L, lead, cx.iloc[0] if len(cx) else None, region)
    rows.append(dict(locus_id=L.locus_id, phenotype=L.phenotype, p=L.p, verdict=fhd.verdict(flags),
                     **{f["check"]: f["status"] for f in flags}))
review = pd.DataFrame(rows)
review.to_csv("felix_dossier_review.tsv", sep="\t", index=False)
display(Markdown("### Verdict over the dossier loci"))
print(review.verdict.value_counts().to_string())
for c in [c for c in review.columns if c not in ("locus_id", "phenotype", "p", "verdict")]:
    print(f"  {c:24s} " + ", ".join(f"{k} {v}" for k, v in review[c].value_counts().items()))
display(review[review.verdict.isin(["fail", "warn"])].head(20))

## 9 · Bundle for download

In [ ]:
!rm -f felix_dossier.tar.gz felix_dossier.tar
!tar -cf felix_dossier.tar felix_dossier_*.tsv dossier_regions/
!gzip felix_dossier.tar
!gsutil cp felix_dossier.tar.gz $WORKSPACE_BUCKET/scratch/kvg/
!ls -la felix_dossier.tar.gz && tar -tzf felix_dossier.tar.gz | head -20

## Caveats

- **The screen comes first.** Every locus here already passed `hit_report`, so the plate reviews survivors, not the
  5,774 excluded hits. The exclusion counts are in `felix_sweep_all_hits.tsv`; the plate's header reports them so the
  reader can see what fraction of the scan reached this document.
- **Only the top-50 rows of each bundle are screenable.** A hit outside a run's `top_hits.tsv` is invisible to both the
  screen and this dossier, and those concentrate in the lowest-case-count runs.
- **No read pileups.** Everything here is derived from the association tables. A locus that passes every check still
  has not been looked at directly, and for the handful that survive, the pileup is the next step, not the last one.
- **The repeat-dosage (`ru`) arm has no genotype counts.** Its `N_*_het/hom` columns hold dosage sums, so
  Hardy-Weinberg, carrier fractions and the fixed-heterozygote check do not apply to it and report `n/a`.
- **Call rate cannot come from FELIX's own input, but it is not out of reach.** `phase_vcfs` is
  `flare_lai_prod.gt_vcf`, the ligated phased callset, and statistical phasing fills every genotype: `MissingRate` is
  exactly 0 in all 331,518 cells of the five region extracts, so the page reports it as not assessed rather than as a
  pass. The call rate, and the fraction of genotypes that phasing **imputed** rather than observed, live in the
  pre-phasing DeepVariant/GLnexus callset, which `propagate_annotations/wdl/AnnotateFlareGqDp.wdl` already reaches
  (it copies `FORMAT/GQ`, `DP` and `RNC` onto FLARE sites, region-indexed). 88 of the 90 leads are SNV or indel and so
  are answerable that way; 1 SV lead needs the SV callset and 1 repeat lead the repeat branch.
- **Context is hg38 UCSC annotation**, not a mappability or coverage track: "unique sequence" here means no overlapping
  segmental duplication record, which is weaker than a mappability estimate from the actual reads.